# Day 49 Practice: Overfitting, Precision/Recall and Leakage

Run the setup cell first (same split as all week). Use `random_state=42` for every model and split.

✅ Exercises 1–4: everyone · Exercise 5: most of you · Exercise 6: challenge

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

DATA_DIR = Path("../../../datasets")
df = pd.read_csv(DATA_DIR / "students.csv")
df["has_job_num"] = (df["has_job"] == "yes").astype(int)

three = ["study_hours", "previous_score", "attendance_pct"]
X_train, X_test, y_train, y_test = train_test_split(df[three], df["passed"], test_size=0.2, random_state=42)
print(len(X_train), len(X_test))

## Exercise 1: Diagnose the experiment

For each result, write **underfitting**, **overfitting**, **good fit** or **suspicious (leakage?)** and give a
one-line reason. The baseline accuracy is 0.65.

| Experiment | Train | Test | Diagnosis | Reason |
|------------|-------|------|-----------|--------|
| A: tree, depth 1 | 0.62 accuracy | 0.60 accuracy | | |
| B: tree, no depth limit | 1.00 accuracy | 0.70 accuracy | | |
| C: logistic regression | 0.84 accuracy | 0.81 accuracy | | |
| D: tree, depth 2 | 1.00 accuracy | 1.00 accuracy | | |
| E: price model (regression) | MAE 0.5 thousand | MAE 31 thousand | | |

## Exercise 2: Your own depth experiment

Using the setup's `X_train`/`X_test` (three features):

1. For each `max_depth` from 1 to 10, train a tree and store `max_depth`, training accuracy and test accuracy
   (rounded to 3 decimals) as a dict in a list. Turn the list into a DataFrame and show it.
2. Plot both accuracies against `max_depth` (two lines with `marker="o"`, labels, legend, title).
3. In the markdown cell: from which depth do you see overfitting? What happens to the test accuracy?

Expected first rows:
```
   max_depth  train_accuracy  test_accuracy
0          1           0.727          0.650
1          2           0.828          0.733
```

In [ ]:
# your code here

*Overfitting starts at depth … because …*

## Exercise 3: Precision and recall by hand

The school's goal (from the demo): flag students who will **fail** (class 0) so a tutor can help.
A depth-3 tree with all six features gives:

```
confusion_matrix(y_test, tree_predictions)
[[12  9]
 [ 5 34]]
```

1. **By hand** (code cell, using the four numbers): recall and precision for the class **fail**.
2. Check with `recall_score` / `precision_score` (`pos_label=0`): train `DecisionTreeClassifier(max_depth=3,
   random_state=42)` on all six features (re-split `df[features6]` with the same settings).
3. The logistic model in the demo had recall 0.71 and precision 0.83. Which model would the school prefer and why?

Expected:
```
recall by hand: 0.57   precision by hand: 0.71
```

In [ ]:
features6 = ["study_hours", "attendance_pct", "sleep_hours", "previous_score", "assignments_done", "has_job_num"]

# your code here

*Which model and why:*

## Exercise 4: Leakage hunt

Ken ran this experiment and proudly reports **"best depth 1, accuracy 1.0"**. It contains **three** leaks.

```python
students = pd.concat([df, df.head(50)])     # someone appended the first 50 rows a second time
features = ["study_hours", "previous_score", "final_score", "attendance_pct"]
X_train, X_test, y_train, y_test = train_test_split(
    students[features], students["passed"], test_size=0.2, random_state=42
)
best_depth = 0
best_score = 0
for depth in range(1, 15):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    score = tree.score(X_test, y_test)
    if score > best_score:
        best_depth = depth
        best_score = score
print("best depth", best_depth, "accuracy", best_score)
```

1. Name the three leaks in the markdown cell.
2. Write an honest version: remove duplicates **before** splitting, remove the leaking feature, choose the depth
   on a **validation** split of the training data (`test_size=0.25, random_state=42`), then report the test
   accuracy **once** for the chosen depth.

Expected (honest version): 298 rows after removing duplicates, chosen depth 3, test accuracy 0.767.

*The three leaks:*
1. ...
2. ...
3. ...

In [ ]:
# your honest version here

## Exercise 5: How many students can the tutors take?

The school has only **20** tutoring slots.

1. Train `LogisticRegression(max_iter=1000)` on `features6` (same split as Exercise 3).
2. For the thresholds 0.5 and 0.6 ("flag if P(pass) is below the threshold"), print the number of students
   flagged, the recall and the precision for class 0.
3. Which threshold fits the 20 slots? What does the school give up?

Expected (approximately):
```
threshold 0.5: flagged 18, recall 0.71, precision 0.83
threshold 0.6: flagged 22, recall 0.76, precision 0.73
```

In [ ]:
# your code here

*My recommendation:*

## Exercise 6 (challenge): Choose the depth honestly with cross-validation

In Exercise 2 you may have picked the depth with the best **test** score, which is Leak A.

1. Using only `X_train`/`y_train` (three features), compute the average 5-fold `cross_val_score` accuracy for
   each depth from 1 to 10.
2. Choose the depth with the highest average. Train it on all of `X_train` and report its test accuracy once.
3. Compare with the test accuracy of the depth you would have picked by peeking at the test set in Exercise 2.
   Which number should you report, and why?

In [ ]:
# your code here

*My answer:*